# 04 — Train residual GATv2

This notebook trains the baseline-preserving residual GATv2 model.

It intentionally keeps the same training protocol as the reference GraphSAGE/residual experiments:

- 1,000 sampled training episodes available to the episodic dataset
- 600 fixed validation episodes
- 10 maximum epochs
- 100 training episodes per epoch
- 20 validation episodes per epoch
- 100 episodes for the final validation report
- early-stopping patience of 3 epochs
- the graph microbatch size and model hyperparameters come from `configs/residual_gatv2_5shot.json`

The final score is:

`CLS logits + α × graph logits`

`α` starts at zero, so training begins from the frozen CLS baseline.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source:", SRC_DIR)


In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)


## Load the GATv2 experiment configuration

The JSON contains the model, graph, optimizer, few-shot, temperature, and microbatch settings.

The remaining training-control values below are held equal to the other training notebooks so the architecture comparison stays controlled.


In [ ]:
CONFIG_PATH = Path("configs/residual_gatv2_5shot.json")
config = json.loads(CONFIG_PATH.read_text())

required_keys = [
    "experiment_name",
    "n_way",
    "k_shot",
    "input_dim",
    "hidden_dim",
    "num_layers",
    "attention_heads",
    "edge_dim",
    "dropout",
    "top_k",
    "graph_temperature",
    "cls_temperature",
    "initial_residual_scale",
    "learning_rate",
    "weight_decay",
    "graph_microbatch_size",
    "train_queries_per_class",
    "eval_queries_per_class",
    "train_seed",
    "val_seed",
]

missing = [key for key in required_keys if key not in config]
if missing:
    raise KeyError(f"Missing required GATv2 config keys: {missing}")

# Same training/evaluation protocol as the reference notebooks.
run_config = {
    **config,
    "train_num_episodes": 1000,
    "val_num_episodes": 600,
    "num_epochs": 10,
    "train_episodes_per_epoch": 100,
    "validation_episodes_per_epoch": 20,
    "final_validation_episodes": 100,
    "early_stopping_patience": 3,
    "max_cached_shards": 6,
}

print(json.dumps(run_config, indent=2))


## Restore cached features and create episodic datasets

This is normally the slowest setup step because the DINOv2 cache is copied from Google Drive to fast local Colab storage.


In [ ]:
from cross_image_glot.storage import restore_feature_splits
from cross_image_glot.data import (
    MiniImageNetFeatureDataset,
    FewShotFeatureEpisodeDataset,
)
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder

restore_feature_splits(
    ["train", "val"],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

train_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "train",
    max_cached_shards=run_config["max_cached_shards"],
)

val_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "val",
    max_cached_shards=run_config["max_cached_shards"],
)

train_episodes = FewShotFeatureEpisodeDataset(
    train_features,
    run_config["n_way"],
    run_config["k_shot"],
    run_config["train_queries_per_class"],
    num_episodes=run_config["train_num_episodes"],
    seed=run_config["train_seed"],
    vary_by_epoch=True,
)

val_episodes = FewShotFeatureEpisodeDataset(
    val_features,
    run_config["n_way"],
    run_config["k_shot"],
    run_config["eval_queries_per_class"],
    num_episodes=run_config["val_num_episodes"],
    seed=run_config["val_seed"],
    vary_by_epoch=False,
)

graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(train_features.metadata["grid_size"]),
    top_k=run_config["top_k"],
    min_similarity=None,
    graph_dtype=torch.float32,
    similarity_device=device,
)

print("Train episodes:", len(train_episodes))
print("Validation episodes:", len(val_episodes))
print("Grid size:", train_features.metadata["grid_size"])


## Build the edge-aware GATv2 residual model

The graph construction stays identical to the GraphSAGE experiment. Only the graph encoder changes.

`PatchGATv2Encoder` uses the existing graph edges and their 5-dimensional edge attributes. The same prototype-cosine readout and CLS-preserving residual wrapper are retained.


In [ ]:
from cross_image_glot.models import (
    PatchGATv2Encoder,
    MeanPrototypeCosineReadout,
    CrossImageGraphMatcher,
    BaselinePreservingResidualMatcher,
)

from cross_image_glot.training import (
    evaluate_residual_dataset,
    load_training_checkpoint,
    make_checkpoint,
    save_checkpoint_atomic,
    save_history,
    train_residual_epoch,
)

encoder = PatchGATv2Encoder(
    input_dim=run_config["input_dim"],
    hidden_dim=run_config["hidden_dim"],
    num_layers=run_config["num_layers"],
    heads=run_config["attention_heads"],
    edge_dim=run_config["edge_dim"],
    dropout=run_config["dropout"],
)

readout = MeanPrototypeCosineReadout(
    temperature=run_config["graph_temperature"],
    learnable_temperature=False,
)

graph_matcher = CrossImageGraphMatcher(
    encoder=encoder,
    readout=readout,
)

model = BaselinePreservingResidualMatcher(
    graph_matcher,
    run_config["initial_residual_scale"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=run_config["learning_rate"],
    weight_decay=run_config["weight_decay"],
)

num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Trainable parameters: {num_trainable:,}")
print("Initial residual scale:", float(model.residual_scale.detach().cpu()))


## Checkpoint and resume state

Checkpoints are stored under the experiment name from the JSON, so this GATv2 run does not overwrite the GraphSAGE or residual-GraphSAGE checkpoints.


In [ ]:
checkpoint_dir = paths.drive_checkpoint_dir / run_config["experiment_name"]
result_dir = paths.drive_results_dir / run_config["experiment_name"]

checkpoint_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)

latest = checkpoint_dir / "latest.pt"
best = checkpoint_dir / "best.pt"

history = []
start_epoch = 0
best_accuracy = float("-inf")
without_improvement = 0

RESUME = True

if RESUME and latest.exists():
    state = load_training_checkpoint(
        latest,
        model,
        optimizer,
        device,
    )
    start_epoch = state["epoch"] + 1
    best_accuracy = state["best_validation_accuracy"]
    without_improvement = state["epochs_without_improvement"]
    history = state.get("history", [])

    print("Resumed at epoch:", start_epoch)
    print("Best validation accuracy:", best_accuracy)
else:
    print("Starting a new training run.")


## Train

Only validation performance is used for model selection and early stopping. The test split is not touched here.


In [ ]:
for epoch in range(start_epoch, run_config["num_epochs"]):
    print(f"\nEpoch {epoch + 1}/{run_config['num_epochs']}")

    train_metrics = train_residual_epoch(
        model,
        optimizer,
        graph_builder,
        train_episodes,
        device,
        epoch,
        run_config["train_episodes_per_epoch"],
        run_config["graph_microbatch_size"],
        run_config["cls_temperature"],
        log_interval=10,
    )

    val_metrics = evaluate_residual_dataset(
        model,
        graph_builder,
        val_episodes,
        device,
        run_config["validation_episodes_per_epoch"],
        run_config["graph_microbatch_size"],
        run_config["cls_temperature"],
        log_interval=5,
        split_name="validation",
    )

    record = {
        "epoch": epoch,
        "train_loss": train_metrics.loss,
        "train_accuracy": train_metrics.accuracy,
        "validation_loss": val_metrics.loss,
        "validation_accuracy": val_metrics.accuracy,
        "residual_scale": float(model.residual_scale.detach().cpu()),
    }
    history.append(record)

    improved = val_metrics.accuracy > best_accuracy

    if improved:
        best_accuracy = val_metrics.accuracy
        without_improvement = 0
    else:
        without_improvement += 1

    checkpoint = make_checkpoint(
        model,
        optimizer,
        epoch,
        best_accuracy,
        without_improvement,
        history,
        run_config,
    )

    save_checkpoint_atomic(checkpoint, latest)

    if improved:
        save_checkpoint_atomic(checkpoint, best)

    save_history(history, result_dir)

    print(record)
    print("Best validation accuracy:", best_accuracy)

    if without_improvement >= run_config["early_stopping_patience"]:
        print("Early stopping.")
        break


## Final validation evaluation

Reload the best checkpoint and evaluate it on 100 fixed validation episodes, matching the reference training notebooks.


In [ ]:
from cross_image_glot.storage import atomic_json_save

state = torch.load(
    best,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(state["model_state_dict"])
model.to(device)
model.eval()

final_metrics = evaluate_residual_dataset(
    model,
    graph_builder,
    val_episodes,
    device,
    run_config["final_validation_episodes"],
    run_config["graph_microbatch_size"],
    run_config["cls_temperature"],
    log_interval=10,
    split_name="validation",
)

validation_result = {
    **final_metrics.to_dict(),
    "residual_scale": float(model.residual_scale.detach().cpu()),
    "experiment_name": run_config["experiment_name"],
}

atomic_json_save(
    validation_result,
    result_dir / "validation_metrics.json",
)

print("Best GATv2 validation metrics:", final_metrics)
print("Residual scale:", model.residual_scale.item())
print("Saved:", result_dir / "validation_metrics.json")
